# Hệ thống tưới cây thông minh ESP32 + ThingSpeak
## Notebook Phân tích dữ liệu & AI

**Đồ án:** Hệ thống tưới cây thông minh sử dụng ESP32 + ThingSpeak
**Nội dung:** Thu thập dữ liệu → Tiền xử lý → Phân tích → Trực quan hóa → Dự báo AI → Quyết định điều khiển bơm

---
Notebook này có thể chạy trực tiếp trên **Google Colab** hoặc **Jupyter/VS Code** cục bộ.
Toàn bộ logic được tổ chức thành các module trong thư mục `src/` để mã nguồn sạch, dễ bảo trì.


## 0. Chuẩn bị môi trường

- Nếu chạy trên **Google Colab**: upload toàn bộ thư mục `src/` (hoặc clone repo) vào Colab trước khi chạy.
- Nếu chạy **cục bộ (VS Code/Jupyter)**: đảm bảo notebook nằm cùng cấp với thư mục `src/`, hoặc chỉnh `SRC_PATH` bên dưới cho đúng.


In [ ]:
# Cài đặt các thư viện cần thiết (Colab thường đã có sẵn, chạy để chắc chắn)
!pip install -q requests pandas numpy matplotlib scikit-learn scipy


In [ ]:
import sys
import os

# Đường dẫn tới thư mục src/ chứa các module của dự án.
# Trên Colab: nếu bạn đã upload thư mục smart-irrigation-ai/, giữ nguyên dòng dưới.
SRC_PATH = os.path.join("..", "src") if os.path.exists(os.path.join("..", "src")) else "src"
sys.path.append(SRC_PATH)

import pandas as pd
import numpy as np

import api
import preprocess
import analysis
import visualization
import prediction
import irrigation_logic

# `display()` chỉ có sẵn mặc định trong môi trường IPython/Jupyter/Colab.
# Định nghĩa fallback để notebook cũng chạy được khi export/run như script thường.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

print("Đã import thành công toàn bộ module của dự án.")


## 1. Thu thập dữ liệu từ ThingSpeak

Thay `CHANNEL_ID` và `READ_API_KEY` bằng thông tin channel ThingSpeak thật của bạn.
Nếu chưa có dữ liệu thật (chưa deploy ESP32), notebook sẽ tự động chuyển sang dùng
**dữ liệu mô phỏng** (500 dòng, sinh bởi `preprocess.generate_simulated_data()`).


In [ ]:
CHANNEL_ID = "YOUR_CHANNEL_ID"          # TODO: thay bằng Channel ID thật
READ_API_KEY = "YOUR_READ_API_KEY"      # TODO: thay bằng Read API Key thật
USE_REAL_DATA = False                    # Đặt True khi đã có channel ThingSpeak thật

if USE_REAL_DATA:
    try:
        raw_df = api.fetch_thingspeak_data(CHANNEL_ID, READ_API_KEY, results=500)
        print(f"Đã lấy {len(raw_df)} dòng dữ liệu thật từ ThingSpeak.")
    except api.ThingSpeakAPIError as e:
        print(f"Lỗi khi gọi API: {e}\nChuyển sang dùng dữ liệu mô phỏng.")
        raw_df = preprocess.generate_simulated_data(n_rows=500)
else:
    raw_df = preprocess.generate_simulated_data(n_rows=500)

raw_df.head()


## 2. Tiền xử lý dữ liệu (Data Cleaning)

In [ ]:
clean_df = preprocess.clean_data(raw_df)
clean_df.to_csv("../data/cleaned_data.csv", index=False)
clean_df.head()


## 3. Phân tích thống kê dữ liệu

In [ ]:
stats = analysis.compute_statistics(clean_df)
analysis.print_statistics(stats)

corr_matrix = analysis.compute_correlation_matrix(clean_df)
print("Ma trận tương quan:")
display(corr_matrix.round(2))

pump_summary = analysis.summarize_pump_activity(clean_df)
print("Tổng hợp hoạt động bơm:", pump_summary)


## 4. Trực quan hóa dữ liệu (7 biểu đồ)

In [ ]:
os.makedirs("../images", exist_ok=True)
visualization.generate_all_charts(clean_df, corr_matrix, output_dir="../images")


## 5. AI dự báo đơn giản: Moving Average + Linear Regression

- **Moving Average**: làm mượt dữ liệu, xác định xu hướng (trend) độ ẩm đất.
- **Linear Regression**: dự báo giá trị độ ẩm đất trong tương lai và ước lượng
  thời điểm cần tưới tiếp theo.


In [ ]:
df_ma = prediction.add_moving_average(clean_df, column="soil_moisture", window=8)

model, metrics = prediction.train_linear_regression(df_ma, column="soil_moisture")
print("Chỉ số đánh giá mô hình Linear Regression:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

is_drying = prediction.detect_drying_trend(df_ma)
print(f"\nXu hướng độ ẩm đất đang giảm liên tục (sắp khô)? {is_drying}")


In [ ]:
forecast_df = prediction.forecast_future(model, df_ma, steps_ahead=96, freq_minutes=15)
next_watering_time = prediction.estimate_time_to_threshold(model, df_ma, freq_minutes=15)

print("Dự báo 96 bước tiếp theo (24 giờ):")
display(forecast_df.head())
print(f"\nThời điểm dự kiến cần tưới tiếp theo: {next_watering_time}")

prediction.plot_prediction(df_ma, forecast_df, save_path="../images/08_prediction_soil_moisture.png")


## 6. Thuật toán quyết định điều khiển bơm (Irrigation Logic)

In [ ]:
# Tính cờ cảnh báo "xu hướng khô" cho từng dòng dựa trên Moving Average (cửa sổ trượt 6 điểm gần nhất)
drying_flags = df_ma["soil_moisture_ma"].rolling(window=6).apply(
    lambda w: all(pd.Series(w).diff().dropna() < 0), raw=False
).fillna(0).astype(bool)

decision_df = irrigation_logic.run_decision_over_dataframe(df_ma, drying_flags=drying_flags)
decision_df[["created_at", "soil_moisture", "decision_action", "decision_warning"]].tail(15)


In [ ]:
# Quyết định tưới tại thời điểm HIỆN TẠI (bản ghi mới nhất)
latest = decision_df.iloc[-1]
final_decision = irrigation_logic.decide_irrigation(
    soil_moisture=latest["soil_moisture"],
    is_drying_trend=is_drying,
)
print("=== QUYẾT ĐỊNH TƯỚI TẠI THỜI ĐIỂM HIỆN TẠI ===")
print(f"Hành động : {final_decision.action.value}")
print(f"Cảnh báo  : {final_decision.warning}")
print(f"Lý do     : {final_decision.reason}")


## 7. Kết luận nhanh

- Toàn bộ dữ liệu, thống kê, biểu đồ (7 biểu đồ + 1 biểu đồ dự báo) đã được lưu vào thư mục `images/` và `data/`.
- Mô hình Linear Regression cho thấy xu hướng biến động của độ ẩm đất theo thời gian.
- Thuật toán quyết định kết hợp ngưỡng tĩnh (40%/60%) và xu hướng Moving Average để đưa ra
  quyết định bật/tắt bơm hợp lý, đồng thời cảnh báo sớm khi đất sắp khô.
- Chi tiết đầy đủ xem tại `report/BaoCao.docx`.
